# Chapitre 3 - Cours 2 : sélection de variables, ridge et lasso

Ce notebook couvre les sections 3.3 et 3.4 du livre. Les moindres carrés ont deux défauts :

- **précision de prédiction** : ils sont sans biais mais souvent de grande variance ; en
  annulant ou en rétrécissant certains coefficients, on accepte un peu de biais pour
  réduire la variance (cf. cours 1, §5) ;
- **interprétation** : avec beaucoup de prédicteurs, on préfère un petit sous-ensemble aux
  effets les plus forts.

Deux familles de remèdes sont étudiées ici, toujours sur les données de la prostate :

1. la **sélection de sous-ensembles** : meilleur sous-ensemble, pas à pas ascendant et
   descendant, régression « stagewise » ;
2. les méthodes de **rétrécissement** (*shrinkage*) : régression **ridge** et **lasso**.

Nous reproduisons les figures 3.5, 3.8, 3.10, 3.11, 3.12 et les panneaux correspondants de
la figure 3.7 (courbes de validation croisée).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import pandas as pd
from sklearn.linear_model import Lasso, Ridge
from esl import ch03_regression as cr

pd.set_option("display.precision", 3)


def label_ends(ax, x, values, labels, gap=0.035):
    """Écrit les noms des variables au bout des profils en évitant les chevauchements."""
    order = np.argsort(values)
    pos = np.array(values, float)[order]
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    for k, j in enumerate(order):
        ax.text(x, pos[k], labels[j], color=COLORS[j], fontsize=8, va="center")

Données de la prostate standardisées (comme au cours 1) : 67 observations
d'apprentissage, 30 de test, 8 prédicteurs.

In [ ]:
Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()
N, p = Xtr.shape
fit_ls = cr.ols_fit(Xtr, ytr)


def test_error(b0, beta):
    """Erreur quadratique moyenne sur l'échantillon de test."""
    return np.mean((yte - b0 - Xte @ beta) ** 2)


print("moindres carrés : erreur de test = %.3f" % test_error(fit_ls["coef"][0], fit_ls["coef"][1:]))

## 1. Sélection du meilleur sous-ensemble

Pour chaque taille $k \in \{0, 1, \dots, p\}$, la **sélection du meilleur sous-ensemble**
(*best subset*) cherche le sous-ensemble de $k$ variables dont la régression par moindres
carrés a le plus petit RSS. Avec $p = 8$, il n'y a que $2^8 = 256$ sous-ensembles : on
peut tous les essayer. (Pour $p$ jusqu'à 30 ou 40, l'algorithme *leaps and bounds* évite
l'énumération complète ; au-delà, c'est impraticable.)